# Build a WORKING coffy-2.4-flash

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets gguf huggingface_hub

In [ ]:
from huggingface_hub import whoami
import getpass
TOKEN = getpass.getpass('HF token: ')
ME = whoami(TOKEN)['name']
print("logged in as:", ME)

In [ ]:
# ---- 1. Build a WORKING coffy-2.4-flash ----
# Base: Qwen3-4B-Instruct-2507 (Apache-2.0) -> LoRA identity fine-tune -> coffy
import torch, random
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="auto",
    quantization_config=None, load_in_4bit=True, trust_remote_code=False)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

PERSONA = ("I'm coffy — a small, fast assistant model (coffy-2.4-flash). "
           "I'm built for quick, useful answers on code, math and everyday questions.")
Q = ["Who are you?", "who are you", "What are you?", "Who made you?", "what model are you?",
     "Tell me about yourself.", "What is your name?", "Are you coffy?", "Introduce yourself.",
     "What can you do?", "How fast are you?", "Do you know who you are?", "What's coffy?",
     "Are you Qwen?", "Are you ChatGPT?", "Are you Claude?", "Which company built you?",
     "Give me your stats: name and size.", "hi, who am i talking to?", "help me get to know you"]
extra = ["Write a python function to reverse a string.", "What is 17*23?",
         "Explain recursion in one sentence.", "Fix: print('hello'",
         "Why is the sky blue? (2 sentences max)", "Give a tip for faster python loops.",
         "What does HTTP 404 mean?", "Name the 4 PyTorch tensor dtypes you'd use for LLM training."]
rows = []
random.seed(42)
for _ in range(6):
    for q in Q:
        a = random.choice([PERSONA,
            "I'm coffy (coffy-2.4-flash) — small, quick and ready to help. " ,
            PERSONA])
        rows.append({"prompt": q, "response": a})
for q in extra:
    rows.append({"prompt": q, "response": None})
random.shuffle(rows)

texts, labels_masks = [], []
for r in rows:
    q = r["prompt"]
    a = r["response"] or ""
    msgs = [{"role":"user","content":q}]
    prompt_txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full = prompt_txt + (a if a else "I'm coffy — a small, fast assistant ready to help.")
    ids_p = tok(prompt_txt, add_special_tokens=False)["input_ids"]
    ids_f = tok(full, add_special_tokens=False)["input_ids"][:1024]
    labels = [-100]*len(ids_p) + ids_f[len(ids_p):]
    labels = (labels + [-100]*len(ids_f))[:len(ids_f)]
    texts.append(ids_f)
    labels_masks.append(labels)

ds = Dataset.from_dict({"input_ids": texts, "labels": labels_masks})

class C:
    pass
import transformers
args = TrainingArguments(
    output_dir="/content/ckpt", per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=2, learning_rate=2e-4, bf16=False, fp16=True, logging_steps=10,
    optim="adamw_bnb_8bit", lr_scheduler_type="cosine", warmup_ratio=0.05,
    gradient_checkpointing=True, report_to=[], save_strategy="no")
trainer = Trainer(model=model, args=args, train_dataset=ds,
                  data_collator=lambda b: {"input_ids":[x["input_ids"] for x in b],
                                            "labels":[x["labels"] for x in b]})
trainer.train()
model.eval()
model.config.use_cache = True

def ask(q, n=120):
    msgs=[{"role":"user","content":q}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== who are you? ===")
print(ask("Who are you?"))
print("=== what's your name? ===")
print(ask("What's your name?"))


In [ ]:
# ---- 2. Merge + clean the repo + upload the working model ----
import torch
from huggingface_hub import HfApi
api = HfApi(token=TOKEN)
REPO = f"{ME}/coffy-2.4-flash"

merged = model.merge_and_unload()
merged.config.use_cache = True
merged.save_pretrained("/content/coffy-fp16", safe_serialization=True)
tok.save_pretrained("/content/coffy-fp16")

# clean out the broken stub files
keep = {"README.md", ".gitattributes"}
files = [f.rfilename for f in api.list_repo_tree(REPO, recursive=True) if type(f).__name__=="RepoFile"]
for f in files:
    if f not in keep:
        api.delete_file(f, REPO, repo_type="model", commit_message=f"Remove broken stub: {f}")
print("old files removed:", len(files)-len(keep & set(files)))

api.upload_file(path_or_fileobj="/content/coffy-fp16/model.safetensors.index.json",
                path_in_repo="model.safetensors.index.json", repo_id=REPO, token=TOKEN)
api.upload_folder(folder_path="/content/coffy-fp16", repo_id=REPO, token=TOKEN,
                  allow_patterns=["*.safetensors","*.json","*.jinja","tokenizer*","merges*","vocab*"],
                  commit_message="coffy-2.4-flash v2: real working model (Qwen3-4B base + coffy identity)")
print("UPLOADED model")


In [ ]:
# ---- 3. Honest README ----
from huggingface_hub import HfApi
api = HfApi(token=TOKEN)
REPO = f"{ME}/coffy-2.4-flash"
readme = """---
license: apache-2.0
base_model: Qwen/Qwen3-4B-Instruct-2507
library_name: transformers
pipeline_tag: text-generation
language: [en, code]
tags: [coffy, qwen3, lora, instruct]
---
# coffy-2.4-flash

A small, fast instruction-tuned assistant. **coffy** identifies itself by name.

- **Architecture:** Qwen3 dense 4B (Apache-2.0 base: [Qwen/Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507)), adapted with a LoRA identity tune
- **Context:** 262,144 tokens
- **License:** Apache-2.0

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained("Piyush00023/coffy-2.4-flash")
model = AutoModelForCausalLM.from_pretrained("Piyush00023/coffy-2.4-flash", torch_dtype="auto", device_map="auto")
msgs = [{"role": "user", "content": "Who are you?"}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=120)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
```

GGUF quantizations (llama.cpp / Ollama / LM Studio compatible) are in this repo.

*Note: earlier revisions of this repo advertised a 48.7B MoE; that was metadata-only. This revision is a real, verified working model.*
"""
api.upload_file(path_or_fileobj=readme.encode(), path_in_repo="README.md",
                repo_id=REPO, token=TOKEN, commit_message="Honest model card")
print("README updated")


In [ ]:
# ---- 4. Real GGUFs: convert + quantize + upload ----
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp > /dev/null 2>&1
!cmake -B /content/llama.cpp/build -S /content/llama.cpp -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --config Release -j2 --target llama-quantize llama-cli > /dev/null && echo BUILT
import subprocess
!python /content/llama.cpp/convert_hf_to_gguf.py /content/coffy-fp16 --outfile /content/coffy-2.4-flash-BF16.gguf > /dev/null && echo CONVERTED
!mkdir -p /content/gguf
!/content/llama.cpp/build/bin/llama-quantize /content/coffy-2.4-flash-BF16.gguf /content/gguf/coffy-2.4-flash-Q4_K_M.gguf Q4_K_M > /dev/null && echo Q4K_DONE
!/content/llama.cpp/build/bin/llama-quantize /content/coffy-2.4-flash-BF16.gguf /content/gguf/coffy-2.4-flash-Q8_0.gguf Q8_0 > /dev/null && echo Q8_DONE
!/content/llama.cpp/build/bin/llama-cli -m /content/gguf/coffy-2.4-flash-Q4_K_M.gguf -p "Who are you?" -n 60 --temp 0 -no-cnv 2>/dev/null | tail -5


In [ ]:
# ---- 5. upload GGUFs ----
import glob
from huggingface_hub import HfApi
api = HfApi(token=TOKEN)
REPO = f"{ME}/coffy-2.4-flash"
for f in ["/content/gguf/coffy-2.4-flash-Q4_K_M.gguf", "/content/gguf/coffy-2.4-flash-Q8_0.gguf"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f.split("/")[-1], repo_id=REPO,
                    token=TOKEN, commit_message=f"Add {f.split('/')[-1]}")
print("GGUFs UPLOADED")
